# Beam search decoding & length normalization

_Deep Learning_

**Don't grab the top word at every step — keep the best few partial sentences, and stop short answers from cheating.**

A sequence model (an RNN or Transformer decoder) picks output words one at a time. We don't want a good first word — we want the most probable whole sentence.

       Greedy decoding takes the single highest-probability word at each step. That is myopic: a great first word can force a bad rest-of-sentence, and greedy can never take it back.

---

This notebook builds beam search up one step at a time: a toy next-token model, greedy decoding, then beam search keeping the best few candidates — and finally a sweep showing the beam only ever helps as it widens. Run each cell and read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy / pure Python

We build a tiny next-token model, decode it **greedily** (always grab the top word), then decode it with **beam search** (keep the best few partial sentences), and compare the two sentences' total log-probabilities.

### Step 1 — Build a toy next-token model

We define a 6-word vocabulary (including an end-of-sequence token) and a random `P(next word | current word)` table. We make stopping a little less eager, normalize each row to a valid probability distribution, and switch to **log space** — log-probabilities add up across a sequence and don't underflow the way tiny products do.

In [ ]:
import numpy as np

# ---- toy next-token model: P(next word | current word) ----
VOCAB = ["the", "cat", "dog", "sat", "ran", "<eos>"]   # <eos> = end of sequence
V, EOS, START, MAXLEN = len(VOCAB), 5, 0, 6

rng = np.random.default_rng(11)            # fixed seed -> reproducible
logits = rng.normal(size=(V, V))           # random transition logits
logits[:, EOS] += -1.5                     # make stopping a bit less eager

P = np.exp(logits)
P /= P.sum(1, keepdims=True)               # rows are valid probability distributions
logP = np.log(P)                           # work in log space (stable, additive)

def show(seq):                             # token ids -> words
    return " ".join(VOCAB[i] for i in seq)

### Step 2 — Greedy decoding (the myopic baseline)

Greedy decoding takes the single highest-probability word at each step and never reconsiders. It's fast but short-sighted: one great first word can lock in a poor rest-of-sentence. We accumulate the log-probability as we go and stop at `<eos>`.

In [ ]:
# ---- greedy: take the single top token each step (myopic) ----
def greedy():
    seq = [START]
    lp = 0.0
    for _ in range(MAXLEN):
        nxt = int(np.argmax(logP[seq[-1]]))    # best next word
        lp += logP[seq[-1], nxt]               # accumulate its log-prob
        seq.append(nxt)
        if nxt == EOS:
            break
    return seq, lp

### Step 3 — Beam search (keep the best B candidates)

Beam search keeps the top `B` partial sequences at every step instead of just one. At each step it expands every live beam by every possible next word, scores the candidates by total log-prob, sets aside any that finished with `<eos>`, and keeps the best `B` to carry forward. At the end it returns the single best sequence.

In [ ]:
# ---- beam search: keep the top B partial sequences each step ----
def beam(B):
    beams = [([START], 0.0)]
    done = []
    for _ in range(MAXLEN):
        cand = []
        for seq, lp in beams:
            if seq[-1] == EOS:                 # finished -> set aside
                done.append((seq, lp))
                continue
            for w in range(V):                 # expand every next word
                cand.append((seq + [w], lp + logP[seq[-1], w]))
        if not cand:
            break
        cand.sort(key=lambda t: t[1], reverse=True)   # by log-prob, high first
        beams = cand[:B]                       # keep only the top B
    allf = done + beams
    allf.sort(key=lambda t: t[1], reverse=True)
    return allf[0]                             # best finished/partial sequence

### Step 4 — Compare greedy vs beam

We run both decoders on the same model and print each one's total log-probability and decoded sentence. Beam search finds a sequence with a higher (less negative) log-prob — it wins precisely because it kept alternatives alive that greedy threw away.

In [ ]:
g_seq, g_lp = greedy()
print(f"greedy        logp={g_lp:7.4f}   {show(g_seq)}")

b_seq, b_lp = beam(3)
print(f"beam (B=3)    logp={b_lp:7.4f}   {show(b_seq)}")

print(f"beam wins by  {b_lp - g_lp:+.4f} log-prob")

# greedy        logp=-6.3567   the cat ran dog ran dog ran
# beam (B=3)    logp=-6.2352   the cat ran cat ran dog ran
# beam wins by  +0.1215 log-prob

## Visualize the data & results

_On the toy model, how does the best sequence log-probability beam search finds change as we widen the beam B from 1 (greedy) up to 8?_

### Step 1 — Rebuild the model and a log-prob-only beam

We recreate the *same* toy model (same seed) so the sweep matches the code above. This `beam(B)` is a trimmed version that returns just the best sequence's log-probability — that single number is all we need to chart how beam width affects quality.

In [ ]:
import numpy as np

VOCAB = ["the", "cat", "dog", "sat", "ran", "<eos>"]
V, EOS, START, MAXLEN = len(VOCAB), 5, 0, 6

rng = np.random.default_rng(11)            # same fixed seed as the CODE tab
logits = rng.normal(size=(V, V))
logits[:, EOS] += -1.5

P = np.exp(logits)
P /= P.sum(1, keepdims=True)
logP = np.log(P)

def beam(B):
    beams = [([START], 0.0)]
    done = []
    for _ in range(MAXLEN):
        cand = []
        for seq, lp in beams:
            if seq[-1] == EOS:
                done.append((seq, lp))
                continue
            for w in range(V):
                cand.append((seq + [w], lp + logP[seq[-1], w]))
        if not cand:
            break
        cand.sort(key=lambda t: t[1], reverse=True)
        beams = cand[:B]
    return max(done + beams, key=lambda t: t[1])[1]   # best log-prob

### Step 2 — Sweep the beam width B

We run beam search at widths 1, 2, 4, and 8 and print the best log-prob found at each. `B = 1` reproduces greedy, and the score is **non-decreasing** as `B` grows — a wider beam can only ever match or beat a narrower one, because it considers a superset of the candidates.

In [ ]:
for B in [1, 2, 4, 8]:
    print(f"B={B}  best logp = {beam(B):.3f}")

# B=1  best logp = -6.357   (= greedy)
# B=2  best logp = -6.235
# B=4  best logp = -5.808
# B=8  best logp = -4.311   (non-decreasing in B)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** With beam width $B = 1$, what does beam search reduce to, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that the beam keeps the top $B$ partial sequences at each step. — _With $B = 1$ you keep exactly one partial sequence._
- At each step you expand that one sequence and keep only the single best continuation. — _Keeping the best one continuation each step is exactly the greedy rule._

**Answer:** It becomes greedy decoding — taking the single highest-probability token at every step. Beam search with $B = 1$ keeps one candidate, so it can never explore an alternative.

</details>

**Problem 2.** A model gives two outputs: a 2-token answer with summed log-prob $-1.5$ and a 5-token answer with summed log-prob $-3.0$. Which wins with no normalization, and which with $\alpha = 1$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- No normalization ($\alpha = 0$): compare the raw sums. — _$-1.5 \gt -3.0$, so the short answer wins. This is the short-sequence bias._
- With $\alpha = 1$: divide each by its length. Short: $-1.5 / 2 = -0.75$. Long: $-3.0 / 5 = -0.60$. — _Now we compare average per-token log-prob._
- Compare: $-0.60 \gt -0.75$. — _The longer answer has the better per-token probability._

**Answer:** No normalization picks the short answer ($-1.5 \gt -3.0$). With $\alpha = 1$ the long answer wins ($-0.60 \gt -0.75$), because per-token it is more probable.

</details>

**Problem 3.** Why do we add log-probabilities instead of multiplying probabilities?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- A sequence's probability is a product of many per-step probabilities, each below 1. — _Multiplying many numbers below 1 gives a vanishingly small result._
- In floating point that tiny product underflows to 0.0, destroying the score. — _You can no longer compare candidates — they all read as 0._
- Take logs: $\log$ of a product is a sum of logs, and $\arg\max$ is unchanged. — _Sums of moderate negative numbers are numerically stable._

**Answer:** Multiplying many sub-1 probabilities underflows to 0.0. Logs turn the product into a stable sum, and because $\log$ is monotonic the most-probable sequence is unchanged.

</details>